# 04.7 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/07-cross-validation/exercises.ipynb)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

ten = pd.DataFrame({
    "row": range(1, 11),
    "days_since_last": [5, 80, 12, 60, 28, 95, 42, 18, 72, 52],
    "churned":         [0,  1,  0,  1,  0,  1,  1,  0,  0,  1],
})
X10, y10 = ten[["days_since_last"]], ten["churned"]
ten

## Exercise 1: two folds by hand

Split the ten rows into **2** folds (rows 1–5 and 6–10). For each fold, predict the test rows with 1-nearest-neighbour on `days_since_last`, compute accuracy, then the mean. Check with `cross_val_score(..., cv=KFold(2))`.

In [ ]:
# your code here

## Exercise 2: leave-one-out

`LeaveOneOut()` makes as many folds as rows: train on 9, test on 1, ten times. Compute the LOO accuracy of 1-NN on the ten rows. Which rows are misclassified?

In [ ]:
# your code here

## Exercise 3: how many folds?

On the real training set, compute the cross-validated accuracy of scaled logistic regression with 3, 5, 10 and 20 folds. How do the mean and standard deviation change? How long does each take?

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")                       # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])
features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"train {len(X_train):,} / test {len(X_test):,}")

In [ ]:
# your code here

## Exercise 4: tree depth by CV

Use 5-fold stratified CV to choose `max_depth` (1 to 10) for a `DecisionTreeClassifier` on the real training set. Then score the chosen tree once on the test set.

In [ ]:
# your code here